# Roteamento

In [29]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM  , ChatOllama 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import Markdown, display

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [30]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


In [34]:

from langchain_core.prompts import ChatPromptTemplate

model =  ChatOllama(model="llama3.2:1b")

prompt = ChatPromptTemplate.from_template('''Você é um professor de matemática de ensino fundamental
capaz de dar respostas muito detalhadas e didáticas. Responda a seguinte pergunta de um aluno:
Pergunta: {pergunta}''')
chain_matematica = prompt | model

prompt = ChatPromptTemplate.from_template('''Você é um professor de física de ensino fundamental
capaz de dar respostas muito detalhadas e didáticas. Responda a seguinte pergunta de um aluno:
Pergunta: {pergunta}''')
chain_fisica = prompt | model

prompt = ChatPromptTemplate.from_template('''Você é um professor de história de ensino fundamental
capaz de dar respostas muito detalhadas e didáticas. Responda a seguinte pergunta de um aluno:
Pergunta: {pergunta}''')
chain_historia = prompt | model

prompt = ChatPromptTemplate.from_template('''{pergunta}''')
chain_generica = prompt | model

In [48]:
from pydantic import BaseModel, Field

prompt = ChatPromptTemplate.from_template('Você deve categorizar a seguinte pergunta: {pergunta}')

class Categorizador(BaseModel):
    """Categoriza as perguntas de alunos"""
    area_conhecimento: str = Field(description=""" 
                                   # 'física', 'matemática' ou 'história'. 
                                   A área de conhecimento da pergunta feita pelo aluno.
                                   Deve ser uma unica palavra  'física', 'matemática' ou 'história'. 
                                   Caso não se encaixe em nenhuma delas, retorne 'outra' """)

parser = PydanticOutputParser(pydantic_object=Categorizador)

model_estruturado = prompt | model.with_structured_output(Categorizador)

model_estruturado.invoke({'pergunta': 'Quando foi a inependencia dos Brasil?'})

Categorizador(area_conhecimento=' história ')

### Criando estrutura de roteamento

In [49]:
from langchain_core.runnables import RunnablePassthrough

chain = RunnablePassthrough().assign(categoria=model_estruturado)
chain.invoke({'pergunta': 'Quando foi a inependencia dos estados unidos?'})

{'pergunta': 'Quando foi a inependencia dos estados unidos?',
 'categoria': Categorizador(area_conhecimento='história')}

In [50]:
def route(input):
    if input['categoria'].area_conhecimento == 'matemática':
        return chain_matematica
    if input['categoria'].area_conhecimento == 'física':
        return chain_fisica
    if input['categoria'].area_conhecimento == 'história':
        return chain_historia
    return chain_generica

In [52]:
chain = RunnablePassthrough().assign(categoria=model_estruturado) | route
resposta_n9 =chain.invoke({'pergunta': 'Quando foi a inependencia dos estados unidos?'})

In [55]:
display(Markdown(resposta_n9.content)) 

A independência dos Estados Unidos foi um processo complexo que envolveu várias etapas e eventos ao longo do século XVIII e XIX. É importante notar que a independência dos Estados Unidos não foi uma ação direta, mas sim o resultado de uma série de movimentos e tensões que culminaram na Declaração de Independência do 4 de julho de 1776, assinada por 56 homens, incluindo os futuros presidentes George Washington e John Adams.

A partir daí, os Estados Unidos seguiram um caminho de estabelecimento como uma república, com a Guerra de Independência (1775-1783) como um dos principais eventos que levaram à independência. Nessa guerra, os Estados Unidos lutaram contra a Monarquia Britânica e, no final, foram derrotados, mas a independência foi reafirmada nas Nações Unidas (1781) e em várias resoluções e tratados.

A independência dos Estados Unidos ocorreu de maneira gradual, com várias revoluções e movimentos políticos importantes ao longo do século XIX. Alguns dos principais eventos que contribuíram para a independência dos Estados Unidos incluem:

* A Constituição de 1787, que estabeleceu os princípios básicos da república federal americana.
* A Constituição do 10 de novembro de 1787, que reforçou os princípios da república federal e estabeleceu a Suprema Corte dos Estados Unidos.
* A Guerra Federal (1812-1815), que foi um conflito entre os Estados Unidos e a França em resposta à invasão francesa de Nova Orleans e a perda de território.
* A Guerra de 1812, que foi um conflito entre os Estados Unidos e o Império Britânico em resposta à invasão britânica de Illinois e Michigan.
* A Guerra Civil Americana (1861-1865), que foi um conflito entre os Estados Unidos e a União do Sul em resposta à secessão dos estados do sul e à Guerra Civil Americana.
* A Guerra de Tróia (1898), que foi um conflito entre os Estados Unidos e a República Mexicana em resposta à invasão mexicana de Panamá e a ocupação militar dos Estados Unidos no Caribe.

Em resumo, a independência dos Estados Unidos foi um processo que envolveu várias etapas e eventos ao longo do século XVIII e XIX, que culminaram na Declaração de Independência do 4 de julho de 1776.

In [58]:
resposta_n10 = chain.invoke({'pergunta': 'por que a divisão por zero é indefinida?'})

In [59]:
display(Markdown(resposta_n10.content))

A divisão por zero é indefinida porque não é possível realizar uma divisão entre um número e zero sem que ocorra um erro ou uma falha no cálculo. Isso ocorre porque, ao tentar dividir um número por zero, o algoritmo de divisão utiliza a propriedade de que o produto de dois números é igual à soma de seus termos, e a soma de dois números é igual à diferença de seus termos.

Para exemplo, considere a divisão de 12 por 3. O algoritmo de divisão tenta calcular 12/3, que seria igual a 4. No entanto, ao tentar calcular a diferença entre 12 e 3, o algoritmo de divisão obtém uma fração negativa, que é não possível. Isso ocorre porque a diferença entre 12 e 3 é 9, que é menor que 12, o que não é representado corretamente pelo algoritmo de divisão.

Além disso, a divisão por zero também pode ocorrer quando o divisor é um número que não pode ser expresso como uma fração, como o número zero ou o número infinito. Nesse caso, o algoritmo de divisão não é capaz de calcular a resposta e o resultado é indefinido.

Em resumo, a divisão por zero é indefinida porque não é possível realizar uma divisão entre um número e zero sem que ocorra um erro ou uma falha no cálculo, ou porque o divisor não pode ser expresso como uma fração, o que não é representado corretamente pelo algoritmo de divisão.